# Cargar_datos.ipynb — Carga de la base de datos

**Proyecto final Machine Learning · Predicción de pago a tiempo de créditos**

En un entorno productivo este paso no existiría como notebook: los datos vivirían en el **Data Warehouse (DWH) o Datalake** de la empresa, materializados en una tabla por otro proceso (ETL). Para este proyecto se usa un dataset de ejemplo no productivo en formato `.csv` (`Base_de_datos.csv`, en la raíz del repositorio).

Este notebook define una **función de carga reutilizable** (`cargar_datos`) que valida que:
1. el archivo exista,
2. no venga vacío,
3. traiga exactamente las columnas esperadas (esquema).

Los demás componentes del pipeline (por ejemplo `comprension_eda.ipynb`) reutilizan esta función en lugar de leer el archivo por su cuenta, para que todos trabajen con la misma fuente y las mismas validaciones.

In [1]:
from pathlib import Path

import pandas as pd

# Permite que otro notebook ejecute este con %run -i sin imprimir todo el resumen
MOSTRAR_RESUMEN = globals().get("MOSTRAR_RESUMEN", True)

## 1. Configuración

Se define la ruta del archivo y el **esquema esperado** (nombres de columnas). Si la tabla de origen cambia (se agrega, elimina o renombra una columna), la carga falla de inmediato con un mensaje claro, en lugar de que el error aparezca más adelante en el modelo.

In [2]:
# El notebook vive en mlops_pipeline/, la base está un nivel arriba
RUTA_DATOS = Path("..") / "Base_de_datos.csv"

COLUMNAS_ESPERADAS = [
    "tipo_credito", "fecha_prestamo", "capital_prestado", "plazo_meses",
    "edad_cliente", "tipo_laboral", "salario_cliente", "total_otros_prestamos",
    "cuota_pactada", "puntaje", "puntaje_datacredito", "cant_creditosvigentes",
    "huella_consulta", "saldo_mora", "saldo_total", "saldo_principal",
    "saldo_mora_codeudor", "creditos_sectorFinanciero", "creditos_sectorCooperativo",
    "creditos_sectorReal", "promedio_ingresos_datacredito", "tendencia_ingresos",
    "Pago_atiempo",
]

COLUMNAS_FECHA = ["fecha_prestamo"]

## 2. Función de carga

In [3]:
def cargar_datos(ruta=RUTA_DATOS, validar_esquema=True):
    """Lee la base de créditos desde CSV y valida su estructura básica.

    Parámetros
    ----------
    ruta : str | Path
        Ruta del archivo CSV.
    validar_esquema : bool
        Si es True, verifica que las columnas sean exactamente las esperadas.

    Retorna
    -------
    pandas.DataFrame
        Datos crudos (sin limpieza).
    """
    ruta = Path(ruta)
    if not ruta.exists():
        # Respaldo: si se ejecuta desde la raíz del repositorio
        alternativa = Path("Base_de_datos.csv")
        if alternativa.exists():
            ruta = alternativa
        else:
            raise FileNotFoundError(f"No se encontró la base de datos en: {ruta.resolve()}")

    datos = pd.read_csv(ruta, parse_dates=COLUMNAS_FECHA, encoding="utf-8")

    if datos.empty:
        raise ValueError("La base de datos está vacía.")

    if validar_esquema:
        faltantes = set(COLUMNAS_ESPERADAS) - set(datos.columns)
        sobrantes = set(datos.columns) - set(COLUMNAS_ESPERADAS)
        if faltantes or sobrantes:
            raise ValueError(
                f"El esquema no coincide. Faltan: {sorted(faltantes)} | Sobran: {sorted(sobrantes)}"
            )
        datos = datos[COLUMNAS_ESPERADAS]

    return datos

## 3. Ejecución de la carga

In [4]:
df = cargar_datos()

if MOSTRAR_RESUMEN:
    print(f"Archivo cargado: {RUTA_DATOS}")
    print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")
    print(f"Rango de fechas: {df['fecha_prestamo'].min():%Y-%m-%d} a {df['fecha_prestamo'].max():%Y-%m-%d}")

Archivo cargado: ../Base_de_datos.csv
Filas: 10,763 | Columnas: 23
Rango de fechas: 2024-11-26 a 2026-04-26


In [5]:
if MOSTRAR_RESUMEN:
    display(df.head())

,tipo_credito,fecha_prestamo,capital_prestado,plazo_meses,edad_cliente,tipo_laboral,salario_cliente,total_otros_prestamos,cuota_pactada,puntaje,...,saldo_mora,saldo_total,saldo_principal,saldo_mora_codeudor,creditos_sectorFinanciero,creditos_sectorCooperativo,creditos_sectorReal,promedio_ingresos_datacredito,tendencia_ingresos,Pago_atiempo
0,4,2025-01-07 14:40:36,1852560.0,12,32,Empleado,3500000,1000000,128650,95.227787,...,0.0,0.0,NaN,NaN,0,0,0,916148.0,Creciente,1
1,4,2025-01-09 11:18:48,3181080.0,6,34,Empleado,5000000,2000000,441817,95.227787,...,0.0,0.0,NaN,NaN,0,0,0,4473774.0,Creciente,1
2,9,2024-12-26 18:52:57,670200.0,5,34,Independiente,5000000,2000000,108632,95.227787,...,0.0,274561.0,274561.0,NaN,2,0,1,NaN,NaN,1
3,9,2024-12-04 14:20:50,506806.8,2,25,Independiente,3000000,900000,199684,95.227787,...,0.0,27564.0,27564.0,NaN,1,0,6,NaN,NaN,1
4,4,2025-04-30 18:41:57,999780.0,10,26,Empleado,2000000,600000,92509,95.227787,...,0.0,0.0,NaN,NaN,0,0,0,61000.0,Creciente,1


In [6]:
if MOSTRAR_RESUMEN:
    df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10763 entries, 0 to 10762
Data columns (total 23 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   tipo_credito                   10763 non-null  int64         
 1   fecha_prestamo                 10763 non-null  datetime64[us]
 2   capital_prestado               10763 non-null  float64       
 3   plazo_meses                    10763 non-null  int64         
 4   edad_cliente                   10763 non-null  int64         
 5   tipo_laboral                   10763 non-null  str           
 6   salario_cliente                10763 non-null  int64         
 7   total_otros_prestamos          10763 non-null  int64         
 8   cuota_pactada                  10763 non-null  int64         
 9   puntaje                        10763 non-null  float64       
 10  puntaje_datacredito            10757 non-null  float64       
 11  cant_creditosvigentes     

## 4. Resultado

- La base se carga correctamente: **10.763 registros y 23 columnas**, con el esquema esperado.
- `fecha_prestamo` se interpreta directamente como fecha (`datetime64`).
- Se observa que `tendencia_ingresos` llega como texto, pero tiene valores numéricos mezclados; eso y el resto de problemas de calidad se analizan y corrigen en **`comprension_eda.ipynb`**.
- Este notebook **no modifica** los datos: entrega la base cruda. La limpieza es responsabilidad de las siguientes etapas del pipeline.